In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
doc = documents[0]
doc
print(doc["content"])
print(doc["filename"])


# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts 

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
import json

user_prompt = json.dumps(doc)

user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)

questions=['What problem does retrieval-augmented generation solve for a language model?', 'Why do the lessons use the LLM as a black box instead of digging into how it works?', 'What are the main limits of an LLM that the lesson points out?', 'What will be built in this module as the example RAG application?', 'What topics are covered in the first part of the module, and how is the second part different?']


In [12]:
print(result.questions)

['What problem does retrieval-augmented generation solve for a language model?', 'Why do the lessons use the LLM as a black box instead of digging into how it works?', 'What are the main limits of an LLM that the lesson points out?', 'What will be built in this module as the example RAG application?', 'What topics are covered in the first part of the module, and how is the second part different?']


In [13]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What exactly is retrieval-augmented generation, and why does it help with answers the model wouldn’t know on its own?', 'Why do we treat the language model like a black box in this course instead of building or training one ourselves?', 'What are the main limits of an LLM that make RAG useful, like outdated knowledge or missing access to private data?', 'What are we going to build in this module, and how does the FAQ assistant for the course work?', 'What’s the difference between the first part and the second part of this module, especially the move toward an agent that chooses what to search?']


In [14]:
print(usage)

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=137, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1157)


In [15]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.0007650000000000001,
 'output_cost': 0.0006165,
 'total_cost': 0.0013815}

In [16]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

[{'question': 'What exactly is retrieval-augmented generation, and why does it help with answers the model wouldn’t know on its own?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why do we treat the language model like a black box in this course instead of building or training one ourselves?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main limits of an LLM that make RAG useful, like outdated knowledge or missing access to private data?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are we going to build in this module, and how does the FAQ assistant for the course work?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What’s the difference between the first part and the second part of this module, especially the move toward an agent that chooses what to search?',
  'document': '01-agentic-rag/lessons/01-intro.md'}]

In [17]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [18]:
generate_ground_truth(doc)

([{'question': 'What is the main idea behind retrieval-augmented generation, and how does it help with LLM limitations?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does the course use LLMs as black boxes instead of teaching how they work internally?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What problems do large language models have that make RAG useful?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What kind of example project is this module going to build with RAG?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What topics are covered in the first part of the module, and how is part 2 different?',
   'document': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1125))

In [19]:
# Question 1
target_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

generated_records = []

for doc in documents:
    if doc["filename"] in target_filenames:
        ground_truth, usage = generate_ground_truth(doc)
        generated_records.extend(ground_truth)
        print(doc["filename"], usage)

generated_records

01-agentic-rag/lessons/01-intro.md ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1111)
01-agentic-rag/lessons/02-environment.md ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1403)
01-agentic-rag/lessons/03-rag.md ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1858)


[{'question': 'What problem does retrieval-augmented generation solve for large language models?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG system in plain Python instead of using a framework right away?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are some limits of LLMs that make RAG useful?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What is the FAQ agent in this module supposed to do?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the first part of the module cover before the agentic version?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module, and is Python plus Jupyter enough?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'question': 'How do I set up the project from nothing using uv, and which packages should I add?',
  'document': '01-agentic-

In [20]:
import pandas as pd

# Load CSV into a DataFrame
df = pd.read_csv("data/ground-truth.csv")

# Convert DataFrame rows into a list of dictionaries
ground_truth = df.to_dict(orient="records")

In [21]:
print(ground_truth[0])

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [23]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [24]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [26]:
from embedder import Embedder

embed = Embedder()

In [28]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [chunk["filename"] for chunk in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

295

In [29]:
import numpy as np
X = np.array(vectors)

X.shape

(295, 384)

In [30]:
q = ground_truth[0]["question"]

query_vector = embed.encode(q)


In [100]:
def vector_search(query, num_results=10):
    from minsearch import VectorSearch

    vindex = VectorSearch(keyword_fields=["course"])
    vindex.fit(X, chunks)

    results = vindex.search(query,
                         num_results=num_results)

    results
    return results

In [92]:
def text_search(query, num_results=10):
    from minsearch import Index

    index = Index(
        text_fields=['content'],
        keyword_fields=['course']
    )
    index.fit(chunks)

    search_results = index.search(
        query,
        boost_dict={"question": 2.0, "section": 0.5},
        num_results=num_results
    )

    search_results

    return search_results

In [93]:
question = ground_truth[0]["question"]

t_filenames = text_search(question, num_results=5)
t_filenames

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [101]:
question = ground_truth[0]["question"]
query_vector = embed.encode(question)

t_filenames = vector_search(query_vector)
t_filenames

[{'start': 1000,
  'content': ' LLM, asking it to create questions for each exercise:\n\n```python\nprompt1_template = """\nYou are a fitness expert generating evaluation questions.\nFor the exercise below, generate 2 questions that a user might ask.\nReturn the result as a JSON array with objects containing \'id\' and \'question\' fields.\n\nExercise: {exercise_name}\nActivity type: {type_of_activity}\nEquipment: {type_of_equipment}\nBody part: {body_part}\nMuscle groups: {muscle_groups_activated}\nInstructions: {instructions}\n""".strip()\n\nfrom tqdm.auto import tqdm\nimport json\n\ndf = pd.read_csv("data/data.csv")\n\nresults = []\n\nfor _, row in tqdm(df.iterrows(), total=len(df)):\n    prompt = prompt1_template.format(**row.to_dict())\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=[{"role": "user", "content": prompt}]\n    )\n    questions = json.loads(response.output_text)\n    for q in questions:\n        q["id"] = row["id"]\n     

In [50]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [118]:
def hybrid_search(query, k=60,num_results=10):
    text_results = text_search(query, num_results=10)
    query_vector = embed.encode(query)
    vector_results = vector_search(query_vector, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

In [107]:
question = "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"
hybrid_search(question, k=60)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [53]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [54]:
q = ground_truth[0]
q

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [55]:
doc_id = q["filename"]
text_results = text_search(query=q["question"],num_results=5)

In [56]:
text_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [57]:
for d in text_results:
    print(f'{d["filename"]} == {doc_id}: {d["filename"] == doc_id}')

01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/01-intro.md == 01-agentic-rag/lessons/01-intro.md: True


In [77]:
doc_id = q["filename"]
query_vector = embed.encode(q["question"])
vector_results = vector_search(query_vector,num_results=5)



In [78]:
vector_results

[{'start': 1000,
  'content': ' LLM, asking it to create questions for each exercise:\n\n```python\nprompt1_template = """\nYou are a fitness expert generating evaluation questions.\nFor the exercise below, generate 2 questions that a user might ask.\nReturn the result as a JSON array with objects containing \'id\' and \'question\' fields.\n\nExercise: {exercise_name}\nActivity type: {type_of_activity}\nEquipment: {type_of_equipment}\nBody part: {body_part}\nMuscle groups: {muscle_groups_activated}\nInstructions: {instructions}\n""".strip()\n\nfrom tqdm.auto import tqdm\nimport json\n\ndf = pd.read_csv("data/data.csv")\n\nresults = []\n\nfor _, row in tqdm(df.iterrows(), total=len(df)):\n    prompt = prompt1_template.format(**row.to_dict())\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=[{"role": "user", "content": prompt}]\n    )\n    questions = json.loads(response.output_text)\n    for q in questions:\n        q["id"] = row["id"]\n     

In [79]:
for d in vector_results:
    print(f'{d["filename"]} == {doc_id}: {d["filename"] == doc_id}')

07-project-example/lessons/02-evaluating-retrieval.md == 01-agentic-rag/lessons/01-intro.md: False
07-project-example/lessons/02-evaluating-retrieval.md == 01-agentic-rag/lessons/01-intro.md: False
07-project-example/lessons/02-evaluating-retrieval.md == 01-agentic-rag/lessons/01-intro.md: False
07-project-example/lessons/02-evaluating-retrieval.md == 01-agentic-rag/lessons/01-intro.md: False
07-project-example/lessons/02-evaluating-retrieval.md == 01-agentic-rag/lessons/01-intro.md: False


In [80]:
relevance = []

for d in text_results:
    relevance.append(int(d["filename"] == doc_id))

relevance

[0, 0, 0, 0, 1]

In [63]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [64]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [65]:
relevance_total_text = compute_relevance_total_text(ground_truth)



  0%|          | 0/360 [00:00<?, ?it/s]

In [66]:
relevance_total_text

[[0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 1, 0, 1, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
 [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 1, 0, 1, 0, 0, 0, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
 [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
 [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
 [1, 0, 0,

In [81]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    # add if else based on search function..text and vector
    if search_function == text_search:
        results = search_function(query=q["question"])
    elif search_function == vector_search:
        query_vector = embed.encode(q["question"])
        results = search_function(query_vector)
    
    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [68]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [69]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [70]:
relevance_total

[[0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 1, 0, 1, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
 [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 1, 0, 1, 0, 0, 0, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
 [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
 [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
 [1, 0, 0,

In [71]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [72]:
# Hit rate with text search
hit_rate(relevance_total)

0.8416666666666667

In [102]:
vector_relevance_total = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [103]:
vector_relevance_total

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0,

In [104]:
# Hit rate with vector search
hit_rate(vector_relevance_total)

0.34444444444444444

In [108]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [109]:
mrr(vector_relevance_total)

0.22847552910052918

In [110]:
mrr(relevance_total)

0.2161574074074074

In [124]:
def compute_relevance_hybrid(q, k, num_results=10):
    doc_id = q["filename"]
    results = hybrid_search(q["question"], k=k, num_results=num_results)
    return [int(d["filename"] == doc_id) for d in results]

In [125]:
def compute_relevance_total_hybrid(ground_truth, ks, num_results=10):
    relevance_by_k = {}
    for k in ks:
        relevance_total = []
        for q in tqdm(ground_truth, desc=f"hybrid k={k}"):
            relevance_total.append(compute_relevance_hybrid(q, k=k, num_results=num_results))
        relevance_by_k[k] = relevance_total
    return relevance_by_k

In [126]:
ks = [1, 50, 100, 200]
hybrid_relevance_by_k = compute_relevance_total_hybrid(ground_truth, ks, num_results=10)

hybrid k=1:   0%|          | 0/360 [00:00<?, ?it/s]

hybrid k=50:   0%|          | 0/360 [00:00<?, ?it/s]

hybrid k=100:   0%|          | 0/360 [00:00<?, ?it/s]

hybrid k=200:   0%|          | 0/360 [00:00<?, ?it/s]

In [127]:
hybrid_relevance_by_k

{1: [[0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
  [1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 1, 0],
  [1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 1, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 1, 0, 1, 0, 1],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
  [0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 1, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
  [1, 0, 0,

In [128]:
for k in ks:
    print(
        f"k={k}: hit_rate={hit_rate(hybrid_relevance_by_k[k]):.4f}, "
        f"mrr={mrr(hybrid_relevance_by_k[k]):.4f}"
    )

k=1: hit_rate=0.8111, mrr=0.5691
k=50: hit_rate=0.8111, mrr=0.5488
k=100: hit_rate=0.8111, mrr=0.5488
k=200: hit_rate=0.8111, mrr=0.5488
